## Notebook guide for using Prokbert Tokenizer

This guide will walk you through the usage of ProkBERT: Tokenizer.

### Step 1: Necessary imports

In [2]:
import yaml
import pathlib
from os.path import join
import os
import sys
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

current_path = str(pathlib.Path(os.getcwd()).parent)
sys.path.append(current_path)

from prokbert_tokenizer import ProkBERTTokenizer

# Replace with your path
abs_path = '/home/ligeti/github/prokbert/src/prokbert'
sys.path.append(abs_path)


from config_utils import *
from sequtils import *

import prokbert

2023-10-26 08:44:22,617 - INFO - Note: NumExpr detected 16 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2023-10-26 08:44:22,619 - INFO - NumExpr defaulting to 8 threads.


### Step 2: Setting up the tokenizer

Once you have all the necessary packages installed and imported, we can start using the tokenizer. We have to initialize it with the desired parameters.

The default parameters can be found here:
        
        'prokbert/blob/main/src/prokbert/configs/sequence_processing.yaml'

In [3]:
# Initialize the params
segmentation_params = {'type': 'random'}
tokenization_params = {'kmer':6, 'shift':2} 
               
# Initialize tokenizer
tokenizer = ProkBERTTokenizer(tokenization_params=tokenization_params, 
                              segmentation_params=segmentation_params,
                              operation_space='sequence')

### Step 3: Using the tokenizer

Now we can use the tokenizer.
First, lets define sequences to work with.

With the second parameter of the 'tokenize' function 'all=True', we can set if we want to have the tokenization of all shifted version of the sequence.

In [19]:
# Tokenize a DNA sequence
sequences = ["AATCANGGAATTATTATCGTT", "AATCACGGAATTATTATCGTT"]
tokens, kmers = tokenizer.tokenize(sequences[1], all=True)

print('Tokens:   ')
for tokenset in tokens:
       print(str(tokenset))
print('___________________')
         
print('Kmers:   ')
for tokenset in kmers:
       print(str(tokenset))
print('___________________')

Tokens:   
[2, 214, 3359, 421, 2580, 248, 3905, 978, 3296, 3]
[2, 843, 1133, 1672, 2113, 980, 3320, 3899, 884, 3]
___________________
Kmers:   
['AATCAC', 'TCACGG', 'ACGGAA', 'GGAATT', 'AATTAT', 'TTATTA', 'ATTATC', 'TATCGT']
['ATCACG', 'CACGGA', 'CGGAAT', 'GAATTA', 'ATTATT', 'TATTAT', 'TTATCG', 'ATCGTT']
___________________


If you choose a spesific lca_shift to be shown, you'll only get the tokens as a result.

In [26]:
tokens = tokenizer.tokenize(sequences[1], lca_shift=0)

print('Tokens starting at position 0:   ')
print(str(tokens))
print('___________________')

tokens = tokenizer.tokenize(sequences[1], lca_shift=1)

print('Tokens starting at position 1:   ')
print(str(tokens))
print('___________________')

Tokens starting at position 0:   
['AATCAC', 'TCACGG', 'ACGGAA', 'GGAATT', 'AATTAT', 'TTATTA', 'ATTATC', 'TATCGT']
___________________
Tokens starting at position 1:   
['ATCACG', 'CACGGA', 'CGGAAT', 'GAATTA', 'ATTATT', 'TATTAT', 'TTATCG', 'ATCGTT']
___________________


We can also restore the original sequence from the tokens if all the parameters are unchanged within the tokenizer. Notice that the last few characters might be cut down due to tokenization process.

In [32]:
tokens, kmers = tokenizer.tokenize(sequences[1], all=True)

print('Restore the original string:')
seq_toks = tokenizer.convert_ids_to_tokens(tokens[0])
print('    ' + ''.join(seq_toks))

Restore the original string:
['AATCAC']
    AATCACGGAATTATTATCGT


The following function help us visualize the overlapping tokenization process.

In [33]:
s = pretty_print_overlapping_sequence(sequences[0], kmers[0], tokenizer.tokenization_params)
print(s)

2023-10-26 09:23:25,943 - INFO - Nr. line to cover the seq:  4


    AATCANGGAATTATTATCGTT
0.  A  A
1.    A  C
2.      T
3.        C


    AATCANGGAATTATTATCGTT
0.   T  G
1.     C  G
2.       A
3.         C


    AATCANGGAATTATTATCGTT
0.    A  A
1.      C  A
2.        G
3.          G


    AATCANGGAATTATTATCGTT
0.     G  T
1.       G  T
2.         A
3.           A


    AATCANGGAATTATTATCGTT
0.      A  A
1.        A  T
2.          T
3.            T


    AATCANGGAATTATTATCGTT
0.       T  T
1.         T  A
2.           A
3.             T


    AATCANGGAATTATTATCGTT
0.        A  T
1.          T  C
2.            T
3.              A


    AATCANGGAATTATTATCGTT
0.         T  G
1.           A  T
2.             T
3.               C




We can also check what tokens will contain a given position's nucleotide. The function gives back all shifted version's tokenizations.

In [36]:
# Get k-mers at position 5
kmer_tokens, positions = tokenizer.get_positions_tokens(sequences[0], 5)

# Analyze k-mers
for i, kmers in enumerate(kmer_tokens):
    print(f"K-mers at position {positions[i]}: {kmers}")

You look for nucleotide N at position 5
K-mers at position [0, 1, 2]: ['AATCAN', 'TCANGG', 'ANGGAA']
K-mers at position [0, 1, 2]: ['ATCANG', 'CANGGA', 'NGGAAT']
